# P0b -- Clay v1.5, its own documented fine-tuning recipe

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Continuation of Phase P0 (see `P0_prithvi_vs_unet_fair_finetune.ipynb` in this same folder,
done 2026-10-02: Prithvi beat a from-scratch U-Net on water/built_up IoU, tied on the
dominant class). This notebook adds Clay v1.5 to that same AOI/labels/split, using Clay's
own documented fine-tuning pattern -- **not** Prithvi's recipe forced onto a different
model. See `Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) -- Plan.md`
for the full track.

**What's different from Clay's own recipe vs. Prithvi's (both verified against real
source/docs, 2026-10-02, not guessed):**
- Clay freezes its encoder entirely and trains only a small conv/pixel-shuffle
  segmentation head (`claymodel.finetune.segment.factory.Segmentor`) -- Prithvi did a
  full fine-tune. Different maintainers, different documented pattern, both followed as
  published rather than forced into one shape.
- Clay's own reference pipeline (`claymodel/finetune/segment/chesapeake_datamodule.py`)
  normalizes pixels with the model's own **fixed, published per-band mean/std** from
  `configs/metadata.yaml` (raw Sentinel-2 L2A digital numbers, e.g. blue mean=1105) --
  not computed from this project's own training tiles, and **not** the `*1e-4` reflectance
  scaling P0's Prithvi cell used. Both are real, documented conventions for their
  respective models.
- Clay's own reference `time`/`latlon` inputs are literally zero placeholders
  (`torch.zeros(4)` each, confirmed in `chesapeake_datamodule.py`) -- not something this
  notebook invented to cut corners; Clay's positional encoding reserves space for them
  but the published fine-tuning example doesn't populate them for this task.
- Optimizer/loss/schedule are Clay's own (`claymodel/finetune/segment/chesapeake_model.py`,
  `configs/segment_chesapeake.yaml`): AdamW, lr=1e-5, weight_decay=0.05, betas=(0.9, 0.95),
  only on the (unfrozen) head parameters; loss `FocalLoss` (multiclass), not Dice;
  `CosineAnnealingWarmRestarts(T_0=100, eta_min=lr*100)`. That `eta_min` (1e-3) being
  *larger* than the base lr (1e-5) looks like an upstream quirk, not a transcription
  error here -- kept faithful to their published config rather than silently corrected,
  per this project's rule of following each model's own recipe as documented.


## 1. Setup

In [ ]:
# NOTE (verified 2026-10-02, not guessed): the "claymodel" PyPI package (v1.5.0,
# https://pypi.org/project/claymodel/) is broken for this use case -- downloaded and
# inspected the actual wheel directly: it ships model.py/factory.py/backbone.py etc.
# as loose top-level modules with NO claymodel/ package wrapper (no claymodel.py,
# no claymodel/__init__.py as a real package), so `import claymodel` fails outright,
# and it doesn't include the finetune/ subpackage (where Segmentor actually lives) at
# all -- confirmed by unzipping the real wheel's RECORD, not assumed from the repo's
# README. Its pinned `torch==2.4.0` would also have forced a pointless downgrade from
# Kaggle's preinstalled 2.10.0. Real fix: clone the GitHub repo directly (same as this
# project's own src/), which has the correct nested claymodel/finetune/segment/
# structure, and only pip-install the small genuinely-missing deps.
!pip install -q timm einops huggingface_hub torchmetrics segmentation-models-pytorch pystac-client planetary-computer stackstac rioxarray lightning
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

# -f: always get the latest fixes, rather than silently reusing a stale clone
!rm -rf repo && git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
!rm -rf clay_repo && git clone -q https://github.com/Clay-foundation/model.git clay_repo
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')


**Running the real pipeline as a subprocess, not inline (same verified reason as P0):**
Kaggle's kernel process already has an old numpy loaded in memory from its own startup,
before this notebook's first cell even runs. If the pip install above ends up upgrading
numpy (any transitive dependency might), importing anything numpy-touching in the
*same* process crashes with a stale-vs-fresh numpy mismatch -- confirmed for real on
Kaggle during P0, with no interactive "restart kernel" possible under headless
`kaggle kernels push` execution. A `subprocess.run(['python', ...])` child process never
had the stale numpy preloaded, so it works regardless of whether this install cell
upgrades numpy or not -- cheap insurance, reusing P0's already-proven fix rather than
re-diagnosing the same failure mode from scratch.

**Update 2026-10-02, real first-run error:** the first version of this notebook
`pip install`ed the `claymodel` PyPI package and did `from claymodel.finetune.segment
.factory import Segmentor`, which failed with `ModuleNotFoundError: No module named
'claymodel'` inside the subprocess. Downloaded and unzipped the actual wheel
(`claymodel-1.5.0-py3-none-any.whl`) to check, rather than guessing a subprocess/PATH
issue: the published PyPI wheel is broken for this use case -- it ships `model.py`,
`factory.py`, etc. as **loose top-level modules with no `claymodel` package wrapper at
all** (confirmed via its own `RECORD`/`top_level.txt`), and it doesn't include the
`finetune/` subpackage (where `Segmentor` actually lives) in the first place. It also
pins `torch==2.4.0` exactly, which would have forced a pointless downgrade from
Kaggle's preinstalled 2.10.0 -- seen for real in the first run's diagnostic print
(`torch 2.4.0+cu121`) before the import even failed. Fixed by cloning
`Clay-foundation/model` directly (same pattern as this project's own `src/`) instead of
installing the PyPI package, and dropping `claymodel` from the pip install line
entirely.

## 2. Write the real pipeline to a script (runs as a subprocess, see note above)

In [ ]:
%%writefile run_p0b.py
import sys
sys.path.insert(0, 'repo')
sys.path.insert(0, 'clay_repo')

import numpy as np
import torch
import torch.nn.functional as F
import lightning as L
import segmentation_models_pytorch as smp
from torch import optim
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import F1Score, MulticlassJaccardIndex
from huggingface_hub import hf_hub_download
import pandas as pd

from src.imagery import get_imagery
from src.landcover import reclassify_worldcover, WATER, BUILT_UP
from src.tiling import tile_array, assign_tile_split

from claymodel.finetune.segment.factory import Segmentor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

# ---- 2. Real data -- identical AOI/labels/split to P0 ----
import pystac_client, planetary_computer, stackstac

AOI = (90.30, 23.75, 90.55, 23.95)  # same 3-district tile as H0-H4, H7, P0
CLAY_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']  # BLUE GREEN RED NIR_BROAD SWIR_1 SWIR_2
# Clay's own published metadata.yaml (configs/metadata.yaml, sentinel-2-l2a platform,
# fetched from Clay-foundation/model on GitHub, 2026-10-02) -- mean/std in raw L2A DN,
# wavelength in micrometers, for exactly these 6 bands (blue/green/red/nir/swir16/swir22;
# metadata.yaml also has rededge1-3 and nir08, not used here since this project's AOI
# imagery only fetches this 6-band set, same choice P0 made).
CLAY_MEAN = np.array([1105., 1355., 1552., 2743., 2388., 1835.], dtype='float32')
CLAY_STD = np.array([1809., 1757., 1888., 1742., 1470., 1379.], dtype='float32')
CLAY_WAVES = torch.tensor([0.493, 0.56, 0.665, 0.842, 1.61, 2.19], dtype=torch.float32)
CLAY_GSD = torch.tensor(10.0)  # sentinel-2-l2a gsd, not NAIP's 1.0

composite, meta = get_imagery(AOI, '2026-01-01/2026-02-28', sensor='sentinel-2',
                               cloud_cover_max=60, max_scenes=1, resolution=10,
                               bands=CLAY_BANDS)
print(meta, composite.shape)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=10, epsg=meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
assert worldcover.shape == composite.shape[1:], 'grid mismatch'

labels_4class = reclassify_worldcover(worldcover)
label_map = np.zeros_like(labels_4class, dtype='int64')
label_map[labels_4class == WATER] = 1
label_map[labels_4class == BUILT_UP] = 2
CLASS_NAMES = ['other', 'water', 'built_up']
print('class balance:', {n: float((label_map == i).mean()) for i, n in enumerate(CLASS_NAMES)})

# ---- 3. Tile + spatial-block split (identical convention to P0) ----
TILE_SIZE = 256

feat_tiles, coords = tile_array(np.nan_to_num(composite.values), tile_size=TILE_SIZE)
label_tiles, _ = tile_array(label_map, tile_size=TILE_SIZE)
test_flags = assign_tile_split(coords, width=composite.shape[-1], test_fraction=0.3)
print(f'{len(feat_tiles)} tiles total: {sum(not f for f in test_flags)} train, {sum(test_flags)} held-out (eastern block)')

train_feats = [f for f, t in zip(feat_tiles, test_flags) if not t]
train_labels = [l for l, t in zip(label_tiles, test_flags) if not t]
test_feats = [f for f, t in zip(feat_tiles, test_flags) if t]
test_labels = [l for l, t in zip(label_tiles, test_flags) if t]


class ClayTileDataset(Dataset):
    """Emits {'pixels', 'label', 'time', 'latlon'} -- Clay's own Segmentor input shape
    (confirmed against claymodel/finetune/segment/chesapeake_datamodule.py), normalized
    with Clay's fixed published per-band mean/std on raw DN -- not P0's 1e-4-scaled
    Prithvi convention."""
    def __init__(self, feats, labels):
        self.feats, self.labels = feats, labels

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx] - CLAY_MEAN[:, None, None]) / CLAY_STD[:, None, None]
        return {
            'pixels': torch.from_numpy(x.astype('float32')),
            'label': torch.from_numpy(self.labels[idx]),
            'time': torch.zeros(4),
            'latlon': torch.zeros(4),
        }


BATCH_SIZE = 8
train_loader = DataLoader(ClayTileDataset(train_feats, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(ClayTileDataset(test_feats, test_labels), batch_size=BATCH_SIZE, num_workers=2)
val_loader = test_loader  # same held-out-block-doubles-as-val convention as P0

# ---- 4. Clay v1.5, via claymodel.finetune.segment -- the verified recipe ----
EPOCHS = 100  # same budget as P0, for a genuinely comparable "same epochs" claim

ckpt_path = hf_hub_download(repo_id='made-with-clay/Clay', filename='v1.5/clay-v1.5.ckpt')
print('checkpoint at:', ckpt_path)


class ClaySegmentor(L.LightningModule):
    """Adapted from claymodel's own ChesapeakeSegmentor reference
    (claymodel/finetune/segment/chesapeake_model.py), pointed at this project's own
    6-band Sentinel-2 AOI/labels instead of the Chesapeake NAIP dataset -- same
    optimizer/loss/schedule, same frozen-encoder Segmentor, different waves/gsd/data."""
    def __init__(self, num_classes, ckpt_path, lr, wd, b1, b2, waves, gsd):
        super().__init__()
        self.save_hyperparameters(ignore=['waves', 'gsd'])
        self.waves = waves
        self.gsd = gsd
        self.model = Segmentor(num_classes=num_classes, ckpt_path=ckpt_path)
        self.loss_fn = smp.losses.FocalLoss(mode='multiclass')
        self.iou_weighted = MulticlassJaccardIndex(num_classes=num_classes, average='weighted')
        self.iou_per_class = MulticlassJaccardIndex(num_classes=num_classes, average='none')
        self.f1 = F1Score(task='multiclass', num_classes=num_classes, average='weighted')
        self.class_names = CLASS_NAMES

    def forward(self, batch):
        return self.model({
            'pixels': batch['pixels'],
            'time': batch['time'],
            'latlon': batch['latlon'],
            'gsd': self.gsd,
            'waves': self.waves,
        })

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            [p for p in self.model.parameters() if p.requires_grad],
            lr=self.hparams.lr, weight_decay=self.hparams.wd,
            betas=(self.hparams.b1, self.hparams.b2),
        )
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=100, T_mult=1, eta_min=self.hparams.lr * 100, last_epoch=-1,
        )
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}

    def _shared_step(self, batch, stage):
        labels = batch['label'].long()
        outputs = self(batch)
        if outputs.shape[-2:] != labels.shape[-2:]:
            outputs = F.interpolate(outputs, size=labels.shape[-2:], mode='bilinear', align_corners=False)
        loss = self.loss_fn(outputs, labels)
        self.log(f'{stage}/loss', loss, on_step=False, on_epoch=True, prog_bar=(stage != 'train'), sync_dist=True)
        if stage != 'train':
            self.log(f'{stage}/iou_weighted', self.iou_weighted(outputs, labels), on_epoch=True, sync_dist=True)
        return outputs, labels, loss

    def training_step(self, batch, _):
        _, _, loss = self._shared_step(batch, 'train')
        return loss

    def validation_step(self, batch, _):
        self._shared_step(batch, 'val')

    def test_step(self, batch, _):
        outputs, labels, _ = self._shared_step(batch, 'test')
        per_class = self.iou_per_class(outputs, labels)
        for name, val in zip(self.class_names, per_class):
            self.log(f'test/iou_{name}', val, on_epoch=True, sync_dist=True)
        self.log('test/f1_weighted', self.f1(outputs, labels), on_epoch=True, sync_dist=True)


clay_model = ClaySegmentor(
    num_classes=3, ckpt_path=ckpt_path,
    lr=1e-5, wd=0.05, b1=0.9, b2=0.95,
    waves=CLAY_WAVES, gsd=CLAY_GSD,
)

clay_trainer = L.Trainer(
    accelerator='gpu', devices=1, precision='bf16-mixed',
    max_epochs=EPOCHS, log_every_n_steps=5,
    callbacks=[L.pytorch.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
    default_root_dir='output/p0b_clay/',
)
clay_trainer.fit(clay_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
clay_test_metrics = clay_trainer.test(clay_model, dataloaders=test_loader)
print(clay_test_metrics)

# ---- 5. Join with P0's already-recorded real result (not re-run, already verified) ----
def tidy(metrics, model_name):
    row = {'model': model_name}
    for k, v in metrics[0].items():
        if 'iou' in k.lower() and 'weighted' not in k.lower():
            name = k.split('/')[-1].lower()
            name = name[4:] if name.startswith('iou_') else name
            row[name] = round(v, 3)
    return row

comparison = pd.DataFrame([
    {'model': 'prithvi_eo_v2_300_tl', 'other': 0.934, 'water': 0.442, 'built_up': 0.284},
    {'model': 'unet_resnet34_scratch', 'other': 0.929, 'water': 0.325, 'built_up': 0.201},
    tidy(clay_test_metrics, 'clay_v1.5'),
])
print(comparison.to_string(index=False))
comparison.to_csv('comparison_p0b.csv', index=False)
print('WROTE comparison_p0b.csv')


## 3. Run it as a subprocess, stream the result back

In [ ]:
import subprocess

proc = subprocess.run(['python', 'run_p0b.py'], capture_output=True, text=True)
print(proc.stdout[-6000:])
print('---stderr (tail)---')
print(proc.stderr[-6000:])
print('returncode:', proc.returncode)
assert proc.returncode == 0, 'run_p0b.py failed -- see stderr above, fix from the real traceback, do not guess'


## 4. Results

In [ ]:
import pandas as pd
comparison = pd.read_csv('comparison_p0b.csv')
print(comparison.to_string(index=False))


## 5. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** The APIs above (`Segmentor`,
  `ChesapeakeSegmentor`'s pattern, `configs/metadata.yaml`'s sentinel-2-l2a band stats,
  the real Hugging Face checkpoint path) are copied from Clay-foundation/model's actual
  source on GitHub and the real, resolvable `made-with-clay/Clay` HF repo (both checked
  2026-10-02, not from memory). Kaggle-specific friction on first run is expected and
  gets fixed from the real traceback, same discipline as P0.
- **`EarlyStopping(patience=15)` on `val/loss`** is this notebook's own judgment call
  (same one P0 made), not part of Clay's published recipe -- worth reconsidering once the
  pipeline is confirmed working, since Clay's own config trains the full 100 epochs.
- **`eta_min = lr * 100` in the scheduler is copied faithfully from Clay's own
  `configs/segment_chesapeake.yaml` / `chesapeake_model.py`**, even though it looks
  backwards (the "minimum" LR ends up 100x *larger* than the base LR) -- not fixed here,
  since the point of this phase is each model's own documented recipe, not a
  second-guessed version of it. If it produces a visibly wrong training curve, that is
  itself worth reporting honestly, not silently patching.
- The P0 rows in the final comparison table are **copied from P0's already-verified real
  result**, not re-run -- re-running Prithvi/U-Net here would just burn GPU time to
  reproduce numbers already on record in `P0_prithvi_vs_unet_fair_finetune.ipynb`.
- Once this runs clean: bring back the actual numbers (not just "it worked") so Phase P1
  (a geographically distinct held-out region) builds on a real three-model result, same
  discipline every other phase in this project has followed.


**Update 2026-10-02, real result (Kaggle kernel version 2, COMPLETE, ~38 min, T4 GPU):**
real three-way comparison, same AOI/labels/spatial-block split as P0:

| model | other | water | built_up |
|---|---|---|---|
| prithvi_eo_v2_300_tl | 0.934 | 0.442 | 0.284 |
| unet_resnet34_scratch | 0.929 | 0.325 | 0.201 |
| clay_v1.5 | 0.936 | 0.345 | 0.224 |

Clay (frozen encoder, small trained head) beats the from-scratch U-Net on both harder
classes (water +6% relative, built_up +11% relative), ties on "other" -- but is clearly
behind Prithvi's full fine-tune on both (water -22%, built_up -21% relative to Prithvi).
This is the expected shape of result: a frozen encoder with a small head adapts less
than a full fine-tune, while still beating a from-scratch baseline. Training stopped
around epoch 44/99 (vs. Prithvi/U-Net's full 100 in P0) -- consistent with this
notebook's own `EarlyStopping(patience=15)` on `val/loss` actually triggering here, not
a crash; no explicit Lightning "did not improve"/stopping message was captured in the
log stream to confirm the exact trigger, so this is inferred from the epoch marker, not
fully confirmed -- worth checking if it matters later. The one real bug this run
surfaced (the broken PyPI `claymodel` wheel, section 2 above) is fixed; next is Phase P1
(a geographically distinct held-out region) with all three models.
